# Lecture 15 · Assignment 3. Beyond five rows: weight scale, more data, new problems

The lecture's network learned five rows perfectly. That does not mean it learned the rule. In this assignment you leave
the five-row comfort zone. You will predict the size of the pre-activations from the weight distribution and change the
initialization, generate more data and find out why the lecture's loss stops working, measure generalization and
extrapolation, set weights **by hand** to prove which networks can represent a rule, and finally make an unfriendly new
problem train.

**What you will do**
1. Predict the spread of pre-activations with a variance calculation, then fix the weight scale
2. Generate data, and find out why a summed loss breaks when the dataset grows
3. Measure test error and extrapolation as the training set grows
4. Hand-set weights for three ReLU-shaped rules, and see that "can represent" is not "will learn"
5. Make a call-option payoff with prices around 100 train, which fails as it stands

Cells marked **Exercise** contain `raise NotImplementedError` for you to replace. Each one is followed by a
**check** cell that prints ✅, ❌ or ⏳. Cells marked **Your answer** want a few sentences of explanation. Those are
graded too, and they matter as much as the code. Run the notebook from top to bottom. A later exercise often uses
the function you wrote in an earlier one. The setup already contains the Leaky ReLU and linear-output code from Assignment 2. Parts 3 to 5 train many networks, so expect about five minutes of run time in total.

In [ ]:
import math, random

def check(name, test):
    """ run a test function; print a tick, a cross, or a reminder if not implemented yet """
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet")
        return
    except Exception as e:
        print(f"❌ {name}: raised {type(e).__name__}: {e}")
        return
    print(f"{'✅' if ok else '❌'} {name}")

import statistics
import matplotlib.pyplot as g

## Setup

The lecture's `Scalar` with the `leaky_relu` operation from Assignment 2, and `Neuron`/`Layer`/`MLP` with `act`, `alpha` and
`linear_output`. Also the reorder data, `compute_loss`, `train` and `dead_neurons`.

In [ ]:
class Scalar:
    def __init__(self, value, label='', oprt='', children=()):
        self.value = value
        self.label = label
        self.oprt = oprt
        self.children = children
        self.der = 0.0
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other)
        out = Scalar(self.value + other.value, '', oprt='+', children=(self, other))
        def _backward():
            self.der += 1.0 * out.der
            other.der += 1.0 * out.der
        out._backward = _backward
        return out

    def __radd__(self, other):          # other + self
        return self + other

    def __rmul__(self, other):          # other * self
        return self * other

    def __pow__(self, n):
        out = Scalar(self.value ** n, label='', children=(self,), oprt=f'**{n}')
        def _backward():
            self.der += n * (self.value ** (n-1)) * out.der
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):          # other - self
        return other + (-self)

    def __mul__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other)
        out = Scalar(self.value * other.value, '', oprt='*', children=(self, other))
        def _backward():
            self.der += other.value * out.der
            other.der += self.value * out.der
        out._backward = _backward
        return out

    def relu(self):
        out = Scalar(0 if self.value < 0 else self.value, '', oprt='ReLU', children=(self,))
        def _backward():
            self.der += (out.value > 0) * out.der
        out._backward = _backward
        return out

    def backward(self):
        visited = set()
        topo = list()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.der = 1.0
        for v in reversed(topo):
            v._backward()

In [ ]:
def leaky_relu(self, alpha=0.01):
    out = Scalar(self.value if self.value > 0 else alpha * self.value, '', oprt='LeakyReLU', children=(self,))
    def _backward():
        self.der += (1.0 if self.value > 0 else alpha) * out.der
    out._backward = _backward
    return out

Scalar.leaky_relu = leaky_relu

In [ ]:
class Neuron:
    def __init__(self, n_inputs, act='relu', initial_bias=None, alpha=0.01):
        self.weights = [Scalar(random.uniform(-1,1), label=f'w{i+1}') for i in range(n_inputs)]
        self.bias = Scalar(random.uniform(-1,1), label='b') if initial_bias is None else Scalar(initial_bias, label='b')
        self.act = act
        self.alpha = alpha
        self.last_pre = None

    def __call__(self, x):
        n = sum(xi * wi for xi, wi in zip(x, self.weights)) + self.bias
        self.last_pre = n.value
        if self.act == 'relu':
            return n.relu()
        if self.act == 'leaky':
            return n.leaky_relu(self.alpha)
        return n                          # linear

    @property
    def params(self):
        return self.weights + [self.bias]


class Layer:
    def __init__(self, n_inputs, n_outputs, **kwargs):
        self.neurons = [Neuron(n_inputs, **kwargs) for _ in range(n_outputs)]

    def __call__(self, x):
        outputs = [n(x) for n in self.neurons]
        return outputs[0] if len(outputs) == 1 else outputs

    @property
    def params(self):
        return [p for neuron in self.neurons for p in neuron.params]


class MLP:
    def __init__(self, n_inputs, output_layers, act='relu', linear_output=False, **kwargs):
        sizes = [n_inputs] + output_layers
        n_layers = len(output_layers)
        self.layers = [Layer(sizes[i], sizes[i+1],
                             act=('linear' if (linear_output and i == n_layers - 1) else act), **kwargs)
                       for i in range(n_layers)]

    def __call__(self, x):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def params(self):
        return [p for l in self.layers for p in l.params]

In [ ]:
# x = [demand, stock, incoming],   y (order) = max(0, demand - stock - incoming)
xs = [
    [3.0, 1.0, 1.0],   # 3 - 1 - 1 =  1
    [4.0, 1.0, 0.0],   # 4 - 1 - 0 =  3
    [2.0, 2.0, 1.0],   # 2 - 2 - 1 = -1  -> 0
    [5.0, 2.0, 1.0],   # 5 - 2 - 1 =  2
    [1.0, 0.0, 2.0],   # 1 - 0 - 2 = -1  -> 0
]
ys = [1.0, 3.0, 0.0, 2.0, 0.0]

def compute_loss(nn):
    ypred = [nn(x) for x in xs]
    return sum((ya - yp)**2 for ya, yp in zip(ys, ypred))

def train(model, lr=0.01, epochs=500, zero_grad=True):
    loss_history = []
    for step in range(epochs):
        loss = compute_loss(model)                       # 1. forward pass
        if zero_grad:
            for p in model.params: p.der = 0.0           # 2. zero grad
        loss.backward()                                  # 3. backward pass
        for p in model.params: p.value += -p.der * lr    # 4. update
        loss_history.append(loss.value)
    return loss_history

In [ ]:
def dead_neurons(model):
    """ per layer: how many neurons had pre-activation <= 0 for every training input """
    alive = {id(n): False for layer in model.layers for n in layer.neurons}
    for x in xs:
        model(x)                              # forward pass fills last_pre
        for layer in model.layers:
            for n in layer.neurons:
                if n.last_pre > 0:
                    alive[id(n)] = True       # fired at least once
    return [sum(not alive[id(n)] for n in layer.neurons) for layer in model.layers]

In [ ]:
def success_count(seeds=range(20), lr=0.01, steps=500, **mlp_kwargs):
    """ how many seeds reach a final training loss below 0.05 on the lecture data """
    ok = 0
    for seed in seeds:
        random.seed(seed)
        model = MLP(3, [4, 4, 1], **mlp_kwargs)
        train(model, lr=lr, epochs=steps)
        ok += compute_loss(model).value < 0.05
    return ok

## Part 1. How big are the pre-activations?

Assignment 2 kept running into the same fact. With inputs up to 5 and weights in [-1, 1], the term `sum x w` easily reaches
+-10, so a neuron's sign is decided by a large random number. Let us make that quantitative.

For a first-layer neuron, `z = w1 x1 + w2 x2 + w3 x3 + b`, where the weights and bias are drawn independently from `U(-1, 1)`
and `x` is a training row. A uniform variable on `(-a, a)` has mean 0 and variance `a^2 / 3`.

### Exercise 1.1. Predict the spread

Write `predicted_preact_std(rows, init='uniform')`. It returns the standard deviation of `z` over the random initialization and
over a uniformly chosen row of `rows`.

* `init='uniform'`: weights and bias from `U(-1, 1)`.
* `init='scaled'`: weights from `U(-1, 1) / sqrt(n_inputs)` and bias still from `U(-1, 1)`.

*Hint: the terms are independent with mean 0, so their variances add. For a product of independent `w` and `x` with `E[w] = 0`,
`Var(w x) = Var(w) E[x^2]`.*

In [ ]:
def predicted_preact_std(rows, init='uniform'):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def measured_preact_std(init_scale, seeds=range(2000)):
    """ empirical std of first-layer pre-activations; init_scale multiplies the weights only """
    pres = []
    for seed in seeds:
        random.seed(seed)
        model = MLP(3, [4, 4, 1])
        for n in model.layers[0].neurons:
            for w in n.weights:
                w.value *= init_scale
        for x in xs:
            model(x)
            pres += [n.last_pre for n in model.layers[0].neurons]
    return statistics.pstdev(pres)

def test_prediction():
    p_u, p_s = predicted_preact_std(xs), predicted_preact_std(xs, 'scaled')
    m_u, m_s = measured_preact_std(1.0), measured_preact_std(1 / math.sqrt(3))
    print(f"   uniform: predicted {p_u:.3f}  measured {m_u:.3f}      scaled: predicted {p_s:.3f}  measured {m_s:.3f}")
    return abs(p_u - m_u) / m_u < 0.03 and abs(p_s - m_s) / m_s < 0.03 and abs(p_u - 2.2657) < 1e-3

check("predicted pre-activation spread matches 2000 random networks", test_prediction)

### Exercise 1.2. A scaled initialization

Add an `init` argument to `Neuron` (and let `MLP` pass it through, which `**kwargs` already does). With `init='scaled'` the weights
are `random.uniform(-1, 1) / math.sqrt(n_inputs)`. The bias is unchanged. As in Assignment 2, the sequence of `random.uniform`
calls must stay exactly the same, so `'scaled'` gives the lecture's weights divided by `sqrt(n_inputs)`.

In [ ]:
class Neuron:
    # YOUR CODE HERE (start from the Neuron above)
    def __init__(self, *args, **kwargs):
        raise NotImplementedError

In [ ]:
def test_scaled():
    random.seed(4)
    u = [p.value for p in MLP(3, [4, 4, 1]).params]
    random.seed(4)
    s = [p.value for p in MLP(3, [4, 4, 1], init='scaled').params]
    expected, k = [], 0
    for n_in, n_out in ((3, 4), (4, 4), (4, 1)):
        for _ in range(n_out):
            expected += [w / math.sqrt(n_in) for w in u[k:k + n_in]] + [u[k + n_in]]
            k += n_in + 1
    return all(abs(a - b) < 1e-15 for a, b in zip(s, expected)) and len(s) == 41

check("init='scaled'", test_scaled)

Now compare success rates on the lecture data (about a minute):

In [ ]:
print("uniform,  ReLU                   :", success_count(), "/ 20")
print("scaled,   ReLU                   :", success_count(init='scaled'), "/ 20")
print("scaled,   ReLU + linear output   :", success_count(init='scaled', linear_output=True, steps=1000), "/ 20")

**Your answer.** Scaling the weights alone helps only a little. Using your formula, explain why: what is the predicted spread with `scaled`, which term in the variance now dominates, and what property of the **inputs** (as opposed to the weights) keeps the pre-activations large? Why does adding the linear output make such a big difference on top? PyTorch draws both weights *and* bias of `nn.Linear` from `U(-1/sqrt(n), 1/sqrt(n))`. What would your formula predict for that?

_(write here)_

## Part 2. More data, and why the summed loss breaks

### Exercise 2.1. A data generator

Write `make_reorder_data(n, seed, lo=(0, 0, 0), hi=(6, 3, 3))`. It returns `(X, Y)` with `n` rows. Each row is
`[demand, stock, incoming]`, where feature `k` is `round(rng.uniform(lo[k], hi[k]), 1)` (in the order demand, stock, incoming),
and the target is `max(0.0, demand - stock - incoming)`. Use a private generator `rng = random.Random(seed)` so that you
**do not disturb** the global `random` state that seeds our networks.

In [ ]:
def make_reorder_data(n, seed, lo=(0, 0, 0), hi=(6, 3, 3)):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def test_data():
    X, Y = make_reorder_data(40, 0)
    ok = len(X) == 40 and all(len(r) == 3 for r in X)
    ok &= all(0 <= r[0] <= 6 and 0 <= r[1] <= 3 and 0 <= r[2] <= 3 for r in X)
    ok &= all(round(v, 1) == v for r in X for v in r)
    ok &= Y == [max(0.0, d - s - i) for d, s, i in X]
    ok &= make_reorder_data(40, 0) == (X, Y) and make_reorder_data(40, 1) != (X, Y)
    ok &= X[0] == [5.1, 2.3, 1.3]
    random.seed(123); a = random.random()
    random.seed(123); make_reorder_data(10, 7); b = random.random()
    return ok and a == b

check("make_reorder_data", test_data)

### Exercise 2.2. A general training loop

Write `fit(model, X, Y, lr=0.01, steps=500, loss='sum')` that trains on any data and returns the loss history.
With `loss='sum'` the loss is `sum (y - yhat)^2` as in the lecture. With `loss='mean'` it is that sum times `1/len(X)`.
Also write `mse(model, X, Y)` returning the **mean** squared error as a plain float, for evaluation.

In [ ]:
def fit(model, X, Y, lr=0.01, steps=500, loss='sum'):
    # YOUR CODE HERE
    raise NotImplementedError

def mse(model, X, Y):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def test_fit():
    random.seed(2); a = MLP(3, [4, 4, 1]); h1 = fit(a, xs, ys, steps=50)
    random.seed(2); b = MLP(3, [4, 4, 1]); h2 = train(b, epochs=50)
    random.seed(2); c = MLP(3, [4, 4, 1]); h3 = fit(c, xs, ys, lr=0.05, steps=50, loss='mean')
    return (h1 == h2 and abs(h3[0] - h1[0] / 5) < 1e-12 and abs(h3[-1] - mse(c, xs, ys)) < 0.05
            and [p.value for p in a.params] == [p.value for p in b.params])

check("fit reproduces the lecture's train(); mean loss is the sum / N", test_fit)

In [ ]:
X40, Y40 = make_reorder_data(40, 0)
for loss, lr in (('sum', 0.01), ('mean', 0.01), ('mean', 0.05)):
    random.seed(0)
    model = MLP(3, [4, 4, 1], act='leaky', linear_output=True)
    h = fit(model, X40, Y40, lr=lr, steps=300, loss=loss)
    preds = [model(x).value for x in X40[:4]]
    print(f"loss={loss:4s} lr={lr:<5} start {h[0]:8.3f}  step 10 {h[10]:8.3f}  end {h[-1]:8.4f}   first predictions {[round(p, 2) for p in preds]}")

**Your answer.** With the sum loss the network stalls at about 59.3 and predicts nearly the same number for every row. (a) Which stuck loss from Assignment 2 is this? Compute it for `Y40` and confirm. (b) The same model, learning rate and loss *formula* work on the 5 lecture rows. Explain precisely why moving to 40 rows breaks them. (c) Why is the mean loss the standard choice?

_(write here)_

## Part 3. Does more data help, and does the network understand the rule?

We hold out two fixed evaluation sets:

* a **test** set of 50 rows from the same ranges as the training data, and
* an **extrapolation** set of 30 rows where demand is between 6 and 10, larger than anything seen in training.

### Exercise 3.1. `evaluate`

Write `evaluate(n_train, seeds=range(3), steps=1000, lr=0.03)`. It builds the training set `make_reorder_data(n_train, 0)`, and
for each seed trains `MLP(3, [4, 4, 1], act='leaky', linear_output=True)` with the **mean** loss. It returns a list of
`(train_mse, test_mse, extrap_mse)` tuples, one per seed.

In [ ]:
X_test, Y_test = make_reorder_data(50, 999)
X_extra, Y_extra = make_reorder_data(30, 998, lo=(6, 0, 0), hi=(10, 3, 3))

In [ ]:
def evaluate(n_train, seeds=range(3), steps=1000, lr=0.03):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
sizes, med = [5, 10, 20, 40], {'train': [], 'test': [], 'extrap': []}
for n in sizes:
    r = evaluate(n)
    for k, key in enumerate(('train', 'test', 'extrap')):
        med[key].append(statistics.median(t[k] for t in r))
    print(f"n_train {n:3d}   " + "   ".join(f"seed {s}: train {a:.4f} test {b:.4f} extrap {c:7.4f}" for s, (a, b, c) in enumerate(r)))

check("evaluate returns one triple per seed", lambda: len(r) == 3 and all(len(t) == 3 for t in r))

for key in med:
    g.plot(sizes, med[key], 'o-', label=key)
g.yscale('log'); g.xlabel('training rows'); g.ylabel('median MSE over 3 seeds'); g.legend()
g.title('more data: train vs test vs extrapolation'); g.show()

In [ ]:
# what did two networks trained on the same 40 rows learn? vary demand, keep stock = incoming = 1
X40, Y40 = make_reorder_data(40, 0)
demands = [d / 2 for d in range(0, 23)]
g.plot(demands, [max(0, d - 2) for d in demands], 'k--', label='true rule')
for seed in (0, 1):
    random.seed(seed)
    model = MLP(3, [4, 4, 1], act='leaky', linear_output=True)
    fit(model, X40, Y40, lr=0.03, steps=1000, loss='mean')
    g.plot(demands, [model([d, 1.0, 1.0]).value for d in demands], label=f'seed {seed}')
g.axvline(6, color='gray', lw=0.8); g.text(6.1, 0.2, 'largest training demand')
g.xlabel('demand'); g.ylabel('predicted order'); g.legend(); g.show()

**Your answer.** (a) Describe how train and test error change from 5 to 40 rows, and explain why 5 rows cannot pin down this network even though it fits them perfectly. (b) In the plot, both seeds are good inside the training range, but one of them goes badly wrong beyond it. Explain what happened in terms of the pieces of a ReLU-type network. Why is a low test error no guarantee of correct extrapolation?

_(write here)_

## Part 4. Set the weights by hand

Training finds weights automatically. Setting them yourself is the clearest way to see what a network *can* represent.
`set_params(model, values)` below writes a flat list of numbers into `model.params` (per neuron: weights, then bias).

### Exercise 4.1

Return a model with hand-set weights that computes each rule **exactly**:

1. `hand_reorder()`: `max(0, d - s - i)` using `MLP(3, [1])`, a single ReLU neuron.
2. `hand_straddle()`: `|p - k|` (the payoff of an option *straddle*, price `p`, strike `k`) using
   `MLP(2, [2, 1], linear_output=True)`.
3. `hand_capped(cap=3.0)`: `min(max(0, d - s - i), cap)` (the warehouse can take at most `cap` units) using
   `MLP(3, [2, 1], linear_output=True)`.

*Hint for 3: draw the target as a function of `z = d - s - i`. Can you write it as the difference of two ReLUs?*

In [ ]:
def set_params(model, values):
    assert len(values) == len(model.params), f"need {len(model.params)} numbers, got {len(values)}"
    for p, v in zip(model.params, values):
        p.value = float(v)
    return model

In [ ]:
def hand_reorder():
    # YOUR CODE HERE
    raise NotImplementedError

def hand_straddle():
    # YOUR CODE HERE
    raise NotImplementedError

def hand_capped(cap=3.0):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def test_hand(model_fn, rule, n_in, lo, hi):
    rng = random.Random(42)
    for _ in range(300):
        x = [rng.uniform(lo, hi) for _ in range(n_in)]
        if abs(model_fn(x).value - rule(*x)) > 1e-9:
            return False
    return True

check("hand_reorder", lambda: test_hand(hand_reorder(), lambda d, s, i: max(0, d - s - i), 3, -5, 10))
check("hand_straddle", lambda: test_hand(hand_straddle(), lambda p, k: abs(p - k), 2, -50, 150))
check("hand_capped", lambda: test_hand(hand_capped(), lambda d, s, i: min(max(0, d - s - i), 3.0), 3, -5, 12)
      and test_hand(hand_capped(5.0), lambda d, s, i: min(max(0, d - s - i), 5.0), 3, -5, 12))

### Can gradient descent find them?

`MLP(3, [2, 1])` can represent the capped rule exactly, because you just did it. Let us see how often training from a random
start finds such a solution, compared with the bigger `MLP(3, [4, 4, 1])`.
Both use Leaky ReLU hidden units. This cell takes about two minutes.

In [ ]:
capped_rule = lambda d, s, i: min(max(0.0, d - s - i), 3.0)
def make_capped(n, seed):
    X, _ = make_reorder_data(n, seed)
    return X, [capped_rule(*x) for x in X]

Xc, Yc = make_capped(30, 0)
Xc_test, Yc_test = make_capped(50, 999)
print(f"capped rows in training data: {sum(y == 3.0 for y in Yc)}/30, zero rows: {sum(y == 0 for y in Yc)}/30")
print(f"hand-set MLP(3, [2, 1]) test MSE: {mse(hand_capped(), Xc_test, Yc_test):.2e}")
for sizes_ in ([2, 1], [4, 4, 1]):
    for seed in range(5):
        random.seed(seed)
        model = MLP(3, sizes_, act='leaky', linear_output=True)
        fit(model, Xc, Yc, lr=0.03, steps=1000, loss='mean')
        print(f"MLP(3, {sizes_}) seed {seed}: train MSE {mse(model, Xc, Yc):.4f}   test MSE {mse(model, Xc_test, Yc_test):.4f}"
              f"   dead {dead_neurons(model)}")

**Your answer.** A network that *can* represent the rule exactly often fails to learn it from a random start, while a larger network usually gets close. Using what you saw in the output (look at the dead counts too), explain why. What does this suggest about why practitioners use networks much larger than the "minimum" size?

_(write here)_

## Part 5. Challenge: a call option

A call option with strike `k` pays `max(0, p - k)` when the price is `p`. It has the same shape as the reorder rule. Here is data
with prices between 80 and 120 and strikes between 90 and 110. First, the obvious approach fails:

In [ ]:
def make_option_data(n, seed):
    rng = random.Random(seed)
    X = [[round(rng.uniform(80, 120), 1), round(rng.uniform(90, 110), 1)] for _ in range(n)]
    return X, [max(0.0, p - k) for p, k in X]

Xo, Yo = make_option_data(30, 0)
Xo_test, Yo_test = make_option_data(50, 1)

for lr in (1e-2, 1e-3, 1e-4):
    finals = []
    for seed in range(3):
        random.seed(seed)
        model = MLP(2, [4, 4, 1], act='leaky', linear_output=True)
        try:
            finals.append(round(fit(model, Xo, Yo, lr=lr, steps=200, loss='mean')[-1], 2))
        except OverflowError:
            finals.append('overflow')
    print(f"raw prices, lr={lr:<6}: final training MSE per seed {finals}")
print("for reference, predicting the mean everywhere gives MSE", round(statistics.pvariance(Yo), 2))

### Exercise 5.1

Make it work. Write `option_pipeline(seed)`. It trains a model on `Xo, Yo` from `random.seed(seed)` and returns the **test MSE on
`Xo_test, Yo_test` measured in the original units** (dollars squared). You may change anything: preprocess the inputs (and/or the
targets), choose the architecture, activation, learning rate and number of steps. You may **not** use the test set for training,
and you may **not** hand-set the weights. It has to be learned.

**Target:** median test MSE over seeds 0 to 4 below `0.5`, with each call taking under 30 seconds.

In [ ]:
def option_pipeline(seed):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
import time

def test_pipeline():
    errs = []
    for seed in range(5):
        t = time.time()
        errs.append(option_pipeline(seed))
        print(f"   seed {seed}: test MSE {errs[-1]:.4f}   ({time.time() - t:.1f} s)")
    print(f"   median {statistics.median(errs):.4f}")
    return statistics.median(errs) < 0.5

check("option pipeline: median test MSE < 0.5", test_pipeline)

**Your answer.** Explain why the raw version fails at **every** learning rate. Consider the size of the pre-activations at initialization (use your formula from Part 1 with inputs near 100) and the size of the gradient on a first-layer weight compared with the gradient on a bias. Then explain what your preprocessing changed. Why must the normalisation statistics come from the training data only?

_(write here)_